In [ ]:
# =====================================================================
# Dynamic-Room Physics Engine
#   Variables: (xc,zc,Lx,Lz,rx,ry) — 6D
#   Constraint: 0 ≤ Lx ≤ rx, 0 ≤ Lz ≤ 3, xc∈[Lx/2, rx-Lx/2], zc∈[Lz/2, 3-Lz/2]
#   BS fixed at (10,-100,-10). Grid step ~0.05m.
#   Hotspot: (rx/2, ry/2), radius = min(rx,ry)/3
# =====================================================================

import torch, numpy as np, math, time
from scipy.stats import gaussian_kde
from scipy.stats.qmc import LatinHypercube

try:
    device = torch.device('cuda'); torch.zeros(1, device=device)
except:
    device = torch.device('cpu')
print(f'Device: {device}')

# Fixed parameters
xBs,yBs,zBs = 10.0, -100.0, -10.0
E=8; dB_s=0.075; lam=0.075; Lp=2; dref=abs(yBs)*1.5; rz=3.0
Pd=40.0; Rth=0.2; Nd=-174.0; Bw=20e6; pm=0.2; nu=5
PB=10**(Pd/10)*1e-3; N0f=10**(Nd/10)*1e-3*Bw
Z_HS=[1.5,1.625,1.75,1.875,2.0]; N_Z=5; STEP=0.05

def gen_rwp(rx,ry,sim_time,dt=10,rng_seed=777):
    rng=np.random; rng.seed(rng_seed)
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=min(rx,ry)/3.0
    p_t,p_s=0.6,0.3; tau_h,tau_n=1.5,0.1; v_h,v_n=0.2,1.0; g_h,g_n=0.6,0.2
    ts=int(sim_time/dt)
    def gt():
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=['normal']*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])]
        d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
        else:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n)
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n)
                    else:ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

_CACHE = {}

def build_room(rx_val,ry_val):
    rx,ry = float(rx_val), float(ry_val)
    nx,ny = int(rx/STEP), int(ry/STEP)
    xv=torch.linspace(0,rx,nx,device=device); yv=torch.linspace(0,ry,ny,device=device)
    Xg,Yg=torch.meshgrid(xv,yv,indexing='ij')
    xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
    gl=[]
    for zh in Z_HS:
        gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1))
    gl=torch.cat(gl,dim=0); Ng=gl.shape[0]
    et=gen_rwp(rx,ry,100000,10,777)
    kde=gaussian_kde(et[:,:2].T,bw_method='scott')
    wxy=kde(xyf.cpu().numpy().T).flatten(); wxy=wxy/wxy.sum()
    gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device); gw=gw/gw.sum()
    np.random.seed(42)
    ps=torch.tensor(2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    eta=torch.tensor(10**((-15+5*np.random.rand(Ng))/10),dtype=torch.float32,device=device)
    na=torch.arange(E,dtype=torch.float32,device=device)
    dyB=torch.tensor(0.0-yBs,device=device)
    v1c=lam/(4*math.pi); v1pc=-(2*math.pi/lam); oE=1/math.sqrt(E)
    return gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE

def outage(X):
    """X: (N,6) — [xc,zc,Lx,Lz,rx,ry]"""
    out=[]
    for i in range(0,len(X),200):
        b=torch.tensor(X[i:i+200],dtype=torch.float32,device=device)
        bo=torch.zeros(len(b),device=device)
        for j in range(len(b)):
            key=(round(b[j,4].item(),2),round(b[j,5].item(),2))
            if key not in _CACHE: _CACHE[key]=build_room(*key)
            gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE=_CACHE[key]
            xc,zc,Lx,Lz=b[j,0],b[j,1],b[j,2],b[j,3]
            xu,yu,zu=gl[:,0],gl[:,1],gl[:,2]
            dx=xc-xBs;dy=dyB;dz=zc-zBs
            Rw=torch.sqrt(dx**2+dy**2+dz**2);th=torch.atan2(dy,dx);ph=torch.acos(dz/Rw)
            kx=torch.sin(ph)*torch.cos(th);kz=torch.cos(ph)
            x1,x2=xc-Lx/2,xc-Lx/2;x3,x4=xc+Lx/2,xc+Lx/2
            z1,z3=zc-Lz/2,zc-Lz/2;z2,z4=zc+Lz/2,zc+Lz/2
            def rd(xs,zs): dd=xs-xBs;dz_=zs-zBs;L=torch.sqrt(dd**2+dyB**2+dz_**2);return dd/L,dyB/L,dz_/L
            ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
            uma=torch.stack([ux1,ux2,ux3,ux4]);uzm=torch.stack([uz1,uz2,uz3,uz4])
            umin=uma.min(0).values;umax=uma.max(0).values;zmin=uzm.min(0).values;zmax=uzm.max(0).values
            dux=xu-xBs;duy=yu-yBs;duz=zu-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2)
            uux=dux/Lu;uuz=duz/Lu
            ix=torch.sigmoid(1000*(uux-umin))*torch.sigmoid(1000*(umax-uux))
            iz=torch.sigmoid(1000*(uuz-zmin))*torch.sigmoid(1000*(zmax-uuz))
            il=ix*iz
            d2x=xu-xc;d2y=yu;d2z=zu-zc;Ru=torch.sqrt(d2x**2+d2y**2+d2z**2)
            t1=d2x/Ru;t2=d2z/Ru
            ax=(1-il)*(kx-t1);az=(1-il)*(kz-t2)
            sx=torch.sinc((math.pi/lam)*Lx*ax);sz=torch.sinc((math.pi/lam)*Lz*az)
            p1=(2*math.pi/lam)*dB_s*na*torch.sin(th)
            a1=oE*torch.complex(torch.cos(p1),torch.sin(p1))
            v1m=v1c/Rw;v1p=v1pc*Rw;v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p))
            H1=v1*a1.conj()
            p2=(2*math.pi/lam)*dB_s*na*torch.sin(tt)
            a2=oE*torch.complex(torch.cos(p2),torch.sin(p2))
            v2m=eta*(lam/(4*math.pi*dref));v2=torch.complex(v2m*torch.cos(ps),v2m*torch.sin(ps))
            H2=v2*a2.conj()
            Ht=math.sqrt(E/Lp)*(H1+H2)
            fm=(Lx*Lz*sx*sz)/(lam*Ru)
            fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
            fc=torch.complex(fm*torch.cos(fp),fm*torch.sin(fp))
            He=Ht*fc;Hw=torch.sum(He)/math.sqrt(E)
            dp=(torch.abs(Hw)**2)*pm*PB;it=(nu-1)*dp;sr=dp/(it+N0f)
            ab=math.pi/3.0;Ses=torch.zeros(1,device=device)
            rn=torch.sqrt((xu-xc)**2+yu**2+(zu-zc)**2+1e-12)
            for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
                dot=torch.abs((xu-xc)*math.cos(a)+yu*math.sin(a))
                cp=dot/rn;ph_b=torch.acos(torch.clamp(cp,0,1))
                Se=(math.pi-torch.clamp(2*ab-ph_b,min=0))/math.pi
                Ses=Ses+torch.clamp(Se,1/3,1)
            sr_se=((Ses/4)*dp)/((Ses/4)*it+N0f)
            bits=(torch.log2(1+sr_se)<Rth).float()
            bo[j]=torch.sum(bits*gw)
        out.append(bo.cpu().numpy());torch.cuda.empty_cache()
    return np.concatenate(out)

print('Dynamic physics engine ready.')

# Test
test=np.array([[2.5,1.5,5.0,0.5,5.0,5.0]])
print(f'Test 5x5: {outage(test)[0]*100:.2f}% | rooms cached: {len(_CACHE)}')
test2=np.array([[5.0,1.5,10.0,0.3,10.0,10.0]])
print(f'Test 10x10: {outage(test2)[0]*100:.2f}% | rooms cached: {len(_CACHE)}')

# ============================================================
# LHS Sampling: 15000 samples in 6D
# ============================================================
N=15000
print(f'\nGenerating {N} samples...')
sampler=LatinHypercube(d=6,seed=123)
U=sampler.random(n=N)
rx_s=5+U[:,0]*15; ry_s=5+U[:,1]*15
# Window params: xc in [Lx/2, rx-Lx/2] — generate Lx first, then xc
Lx_s=0.05+U[:,3]*(rx_s-0.05); Lz_s=0.05+U[:,5]*2.95
xc_s=Lx_s/2+U[:,2]*(rx_s-Lx_s)
zc_s=Lz_s/2+U[:,4]*(3-Lz_s)
Xa=np.column_stack([xc_s,zc_s,Lx_s,Lz_s,rx_s,ry_s])
print(f'Shape: {Xa.shape}')
t0=time.time();Ya=outage(Xa)
print(f'Done in {time.time()-t0:.0f}s')
print(f'Feasible (<10%): {(Ya<0.10).sum()} ({(Ya<0.10).sum()/len(Ya)*100:.1f}%)')
print(f'Rooms cached: {len(_CACHE)}')
cols=['xc','zc','Lx','Lz','rx','ry','outage']
np.savetxt('dynamic_train.csv',np.column_stack([Xa,Ya]),delimiter=',',header=','.join(cols),comments='')

# ============================================================
# Feature engineering: relative + physics-informed features
# ============================================================
xc=Xa[:,0];zc=Xa[:,1];Lx=Xa[:,2];Lz=Xa[:,3];rx=Xa[:,4];ry=Xa[:,5]
# Relative features (scale-invariant)
xc_rel=xc/rx; Lx_rel=Lx/rx
area_ratio=(Lx*Lz)/(rx*3.0)
# Physics-informed: BS (10,-100,-10) to window center (xc, ry, zc)
dx=xc-10.0; dy=ry-(-100.0); dz_=zc-(-10.0)
dist_bs=np.sqrt(dx**2+dy**2+dz_**2)
alpha_x=dx/dist_bs; alpha_z=dz_/dist_bs

Xe=np.column_stack([xc,zc,Lx,Lz,rx,ry,
                     xc_rel,Lx_rel,area_ratio,
                     dist_bs,alpha_x,alpha_z])
cols=["xc","zc","Lx","Lz","rx","ry",
      "xc_rel","Lx_rel","area_ratio",
      "dist_bs_win","alpha_x","alpha_z","outage"]
np.savetxt("dynamic_train_enriched.csv",np.column_stack([Xe,Ya]),delimiter=",",header=",".join(cols),comments="")
print(f"Saved dynamic_train_enriched.csv ({Xe.shape[1]} features)")
print('Saved dynamic_train.csv')
